## Replication: Zhang et al. (2022) ResNet18 + ConvGRU on GIFGIF (7 emotions, 4-frame sequences)

This notebook reproduces the **core setup** described in Zhang et al. (2022):

- Dataset: **GIFGIF** (7 emotions), **300 GIFs per class** (total 2100)
- Input: **4 frames** per GIF, resized to **224×224**
- Backbone: **ResNet-18** (spatial features)
- Temporal module: **ConvGRU** (kernel 3×3)
- Optimizer: **Adam**, lr=1e-4, weight_decay=1e-4
- Batch size: **30**, Epochs: **100**
- Metrics: Accuracy, Precision, Recall, F1 (macro + per-class)

> Must set `GIF_ROOT` to the folder containing the GIF files referenced by the official labels CSV.


In [11]:
# If you run on a fresh environment, uncomment installs.
# !pip -q install torch torchvision pillow pandas numpy scikit-learn tqdm --upgrade

import os, random
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from PIL import Image, ImageSequence, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True  # tolerate partially-downloaded GIFs

from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

print("torch:", torch.__version__)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


torch: 2.10.0+cpu
device: cpu


### 1) Paths + official GIFGIF labels

Set `GIF_ROOT` to the directory that contains the `.gif` files.


In [ ]:
CSV_PATH = r"D:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\gifgif_emotion_labels.csv"

# TODO: change this to the local GIF folder path on your PC when running there.
# Example (Windows raw string):
GIF_ROOT = r"D:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\Data\gifgif-images-v1\gifgif-images"

assert os.path.exists(CSV_PATH), f"Missing CSV: {CSV_PATH}"
assert os.path.isdir(GIF_ROOT), f"GIF_ROOT not found: {GIF_ROOT}"

df = pd.read_csv(CSV_PATH)
df.head(), df.shape


(           gif_id                                           gif_path  \
 0  12f31lpt5QGIWQ  D:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\Data\g...   
 1  12qogXnmccuga4  D:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\Data\g...   
 2   W3hdeQrGv6mqs  D:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\Data\g...   
 3   2Wv76xVF415S0  D:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\Data\g...   
 4   j4LLz8zhCXQoU  D:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\Data\g...   
 
   primary_emotion  primary_score  total_comparisons  pleasure_score  \
 0         disgust             50                197               3   
 1      excitement             37                197              18   
 2        surprise             24                159               6   
 3      excitement             36                109              10   
 4       happiness            108                463              63   
 
    disgust_score  happiness_score  pride_score  excitement_score  ...  \
 0             50                2            3     

### 2) Auto-detect column names (filename + emotion)

This cell tries to detect the filename column and emotion label column automatically.
If it picks the wrong columns, just edit `FILE_COL` and `LABEL_COL` manually.


In [13]:
print(df.columns.tolist())

fname_candidates = [c for c in df.columns if c.lower() in ["gif", "gif_name", "filename", "file", "name", "url"]]
label_candidates = [c for c in df.columns if c.lower() in ["emotion", "label", "category", "class"]]

print("Filename candidates:", fname_candidates)
print("Label candidates:", label_candidates)

FILE_COL = fname_candidates[0] if fname_candidates else df.columns[0]
LABEL_COL = label_candidates[0] if label_candidates else df.columns[1]

print("Using FILE_COL =", FILE_COL)
print("Using LABEL_COL =", LABEL_COL)

df[[FILE_COL, LABEL_COL]].head()


['gif_id', 'gif_path', 'primary_emotion', 'primary_score', 'total_comparisons', 'pleasure_score', 'disgust_score', 'happiness_score', 'pride_score', 'excitement_score', 'embarrassment_score', 'surprise_score', 'sadness_score', 'fear_score', 'satisfaction_score', 'guilt_score', 'contempt_score', 'shame_score', 'anger_score', 'amusement_score', 'contentment_score', 'relief_score']
Filename candidates: []
Label candidates: []
Using FILE_COL = gif_id
Using LABEL_COL = gif_path


,gif_id,gif_path
0,12f31lpt5QGIWQ,D:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\Data\g...
1,12qogXnmccuga4,D:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\Data\g...
2,W3hdeQrGv6mqs,D:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\Data\g...
3,2Wv76xVF415S0,D:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\Data\g...
4,j4LLz8zhCXQoU,D:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\Data\g...


### 3) Zhang et al. 7 classes + sampling 300 per class

Zhang used these 7 emotions:
- happiness
- like/satisfaction
- surprise
- sadness
- anger
- disgust
- fear

This cell normalizes label strings, maps common variants to these 7, and samples **exactly 300 GIFs per class**.


In [14]:
print(df.columns.tolist())
df.head(3)


['gif_id', 'gif_path', 'primary_emotion', 'primary_score', 'total_comparisons', 'pleasure_score', 'disgust_score', 'happiness_score', 'pride_score', 'excitement_score', 'embarrassment_score', 'surprise_score', 'sadness_score', 'fear_score', 'satisfaction_score', 'guilt_score', 'contempt_score', 'shame_score', 'anger_score', 'amusement_score', 'contentment_score', 'relief_score']


,gif_id,gif_path,primary_emotion,primary_score,total_comparisons,pleasure_score,disgust_score,happiness_score,pride_score,excitement_score,...,sadness_score,fear_score,satisfaction_score,guilt_score,contempt_score,shame_score,anger_score,amusement_score,contentment_score,relief_score
0,12f31lpt5QGIWQ,D:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\Data\g...,disgust,50,197,3,50,2,3,3,...,9,7,4,11,20,13,14,6,5,3
1,12qogXnmccuga4,D:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\Data\g...,excitement,37,197,18,7,21,30,37,...,0,0,22,0,6,1,19,12,15,6
2,W3hdeQrGv6mqs,D:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\Data\g...,surprise,24,159,6,12,7,4,7,...,8,7,5,13,4,12,8,5,8,11


In [24]:
# Use the official score columns to define Zhang's 7-class label
Z_CLASSES = ["happiness","satisfaction","surprise","sadness","anger","disgust","fear"]
SCORE_COLS = [f"{c}_score" for c in Z_CLASSES]

# 1) File path and ID
FILE_COL = "gif_path"   # absolute path to the GIF
ID_COL   = "gif_id"

# 2) Make sure score columns exist
missing = [c for c in SCORE_COLS if c not in df.columns]
assert not missing, f"Missing score columns: {missing}"

# 3) Compute Zhang label using argmax over the 7 score columns
scores = df[SCORE_COLS].to_numpy()
df["zhang_label"] = [Z_CLASSES[i] for i in scores.argmax(axis=1)]

# 4) Keep only rows where the path exists (important!)
df["gif_path"] = df["gif_path"].astype(str)
df_ok = df[df["gif_path"].map(os.path.exists)].copy()
print("Rows with existing files:", df_ok.shape)

print("Label counts (before sampling):")
print(df_ok["zhang_label"].value_counts().reindex(Z_CLASSES))

# 5) Sample 300 per class like Zhang
SEED = 42
parts = []
for lab in Z_CLASSES:
    sub = df_ok[df_ok["zhang_label"] == lab]
    if len(sub) < 300:
        raise ValueError(f"Not enough samples for {lab}: {len(sub)} < 300")
    parts.append(sub.sample(n=300, random_state=SEED))

df2100 = pd.concat(parts, ignore_index=True)
print("Final sampled dataset:", df2100.shape)
print(df2100["zhang_label"].value_counts().reindex(Z_CLASSES))



Rows with existing files: (6101, 23)
Label counts (before sampling):
zhang_label
happiness       2000
satisfaction     652
surprise         679
sadness          998
anger            911
disgust          459
fear             402
Name: count, dtype: int64
Final sampled dataset: (2100, 23)
zhang_label
happiness       300
satisfaction    300
surprise        300
sadness         300
anger           300
disgust         300
fear            300
Name: count, dtype: int64


### 4) Train/test split (80/20 stratified)

In [25]:
train_df, test_df = train_test_split(
    df2100,
    test_size=0.2,
    random_state=SEED,
    stratify=df2100["zhang_label"]
)

print("Train:", train_df.shape, "Test:", test_df.shape)
print("Train counts:")
print(train_df["zhang_label"].value_counts().reindex(Z_CLASSES))
print("Test counts:")
print(test_df["zhang_label"].value_counts().reindex(Z_CLASSES))


Train: (1680, 23) Test: (420, 23)
Train counts:
zhang_label
happiness       240
satisfaction    240
surprise        240
sadness         240
anger           240
disgust         240
fear            240
Name: count, dtype: int64
Test counts:
zhang_label
happiness       60
satisfaction    60
surprise        60
sadness         60
anger           60
disgust         60
fear            60
Name: count, dtype: int64


### 5) Dataset: GIF → 4 frames (224×224)

Evenly sample 4 frames if GIF is longer; pad by repeating the last frame if shorter.


In [26]:
from PIL import Image, ImageSequence, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

TARGET_SIZE = (224, 224)
SEQ_LEN = 4

# Train augmentation (light, safe)
train_tf = transforms.Compose([
    transforms.Resize(TARGET_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])


# Test transform (no augmentation)
test_tf = transforms.Compose([
    transforms.Resize(TARGET_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

label2id = {lab:i for i, lab in enumerate(Z_CLASSES)}
id2label = {i:lab for lab,i in label2id.items()}

def load_gif_frames(path, max_frames=60):
    frames = []
    im = Image.open(path)
    for i, fr in enumerate(ImageSequence.Iterator(im)):
        if max_frames is not None and i >= max_frames:
            break
        frames.append(fr.convert("RGB"))
    return frames

def sample_or_pad(frames, seq_len=4):
    if len(frames) == 0:
        frames = [Image.new("RGB", TARGET_SIZE, (0,0,0))]
    if len(frames) >= seq_len:
        idxs = np.linspace(0, len(frames)-1, seq_len).round().astype(int).tolist()
        seq = [frames[i] for i in idxs]
    else:
        seq = frames[:]
        while len(seq) < seq_len:
            seq.append(seq[-1])
    return seq

class GifGifSeqDataset(Dataset):
    def __init__(self, df, transform, seq_len=4):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.seq_len = seq_len
        self.bad = 0

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        lab = r["zhang_label"]
        path = r["gif_path"]   # absolute path from CSV

        try:
            frames = load_gif_frames(path)
            frames = sample_or_pad(frames, seq_len=self.seq_len)
            x = torch.stack([self.transform(fr) for fr in frames], dim=0)  # [T,3,224,224]
        except Exception:
            self.bad += 1
            x = torch.zeros((self.seq_len, 3, 224, 224), dtype=torch.float32)

        y = label2id[lab]
        return x, y, os.path.basename(path)

train_ds = GifGifSeqDataset(train_df, train_tf, seq_len=SEQ_LEN)
test_ds  = GifGifSeqDataset(test_df,  test_tf,  seq_len=SEQ_LEN)

BATCH = 30
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=0, pin_memory=(device=="cuda"))
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=0, pin_memory=(device=="cuda"))

xb, yb, fb = next(iter(train_loader))
print("Batch:", xb.shape, yb.shape, "Files:", fb[:3])
print("Bad GIFs so far (train):", train_ds.bad, " | (test):", test_ds.bad)
print("Example labels:", [id2label[int(i)] for i in yb[:3]])


Batch: torch.Size([30, 4, 3, 224, 224]) torch.Size([30]) Files: ('3VVz4BzVm1spa.gif', 'geYwtodB9AiI0.gif', '1Nb40amazwhyg.gif')
Bad GIFs so far (train): 0  | (test): 0
Example labels: ['anger', 'satisfaction', 'sadness']


### 6) Model: ResNet18 backbone + ConvGRU (3×3) + classifier

In [27]:
class ConvGRUCell(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size=3):
        super().__init__()
        padding = kernel_size // 2
        self.hidden_dim = hidden_dim
        self.conv_z = nn.Conv2d(input_dim + hidden_dim, hidden_dim, kernel_size, padding=padding)
        self.conv_r = nn.Conv2d(input_dim + hidden_dim, hidden_dim, kernel_size, padding=padding)
        self.conv_h = nn.Conv2d(input_dim + hidden_dim, hidden_dim, kernel_size, padding=padding)

    def forward(self, x, h_prev):
        if h_prev is None:
            h_prev = torch.zeros(x.size(0), self.hidden_dim, x.size(2), x.size(3), device=x.device)
        cat = torch.cat([x, h_prev], dim=1)
        z = torch.sigmoid(self.conv_z(cat))
        r = torch.sigmoid(self.conv_r(cat))
        cat_r = torch.cat([x, r * h_prev], dim=1)
        h_hat = torch.tanh(self.conv_h(cat_r))
        h = (1 - z) * h_hat + z * h_prev
        return h

class ConvGRU(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size=3):
        super().__init__()
        self.cell = ConvGRUCell(input_dim, hidden_dim, kernel_size)

    def forward(self, x_seq):
        h = None
        for t in range(x_seq.size(1)):
            h = self.cell(x_seq[:, t], h)
        return h

class ResNetConvGRU(nn.Module):
    def __init__(self, num_classes=7, gru_hidden=256):
        super().__init__()
        base = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.backbone = nn.Sequential(*list(base.children())[:-2])  # [B,512,7,7]
        self.convgru = ConvGRU(input_dim=512, hidden_dim=gru_hidden, kernel_size=3)
        self.pool = nn.AdaptiveAvgPool2d((1,1))
        self.drop = nn.Dropout(p=0.4)
        self.fc = nn.Linear(gru_hidden, num_classes)

    def forward(self, x):
        # x: [B,T,3,224,224]
        feats = []
        for t in range(x.size(1)):
            feats.append(self.backbone(x[:, t]))
        feats = torch.stack(feats, dim=1)      # [B,T,512,7,7]
        h = self.convgru(feats)                # [B,H,7,7]
        v = self.pool(h).flatten(1)
        v = self.drop(v)
        return self.fc(v)


model = ResNetConvGRU(num_classes=7, gru_hidden=256).to(device)
print("Params (M):", sum(p.numel() for p in model.parameters())/1e6)


Params (M): 16.487495


In [28]:
def set_backbone_trainable(model, trainable: bool):
    for p in model.backbone.parameters():
        p.requires_grad = trainable

def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Trainable params before:", count_trainable_params(model))

# Freeze backbone initially
FREEZE_EPOCHS = 5
set_backbone_trainable(model, False)

print("Backbone frozen. Trainable params now:", count_trainable_params(model))


Trainable params before: 16487495
Backbone frozen. Trainable params now: 5310983


### 7) Training (100 epochs) + checkpointing + metrics

Saves the best checkpoint by **macro F1** (you can change to accuracy if you want).


In [ ]:
LR = 1e-4
WD = 1e-4
EPOCHS = 100

# IMPORTANT: create optimizer AFTER freezing so it only optimizes trainable params at first
opt = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=WD)

def run_epoch(model, loader, train=True):
    model.train(train)
    losses = []
    y_true, y_pred = [], []

    for x, y, _fn in tqdm(loader, leave=False):
        x, y = x.to(device), y.to(device)

        if train:
            opt.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train):
            logits = model(x)
            loss = F.cross_entropy(logits, y)

            if train:
                loss.backward()
                opt.step()

        losses.append(loss.item())
        y_true.extend(y.detach().cpu().numpy().tolist())
        y_pred.extend(logits.argmax(1).detach().cpu().numpy().tolist())

    acc = accuracy_score(y_true, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    return float(np.mean(losses)), float(acc), float(p), float(r), float(f1)

history = []

best_f1  = -1.0
best_acc = -1.0

CKPT_F1_PATH  = "zhang_best_f1.pt"
CKPT_ACC_PATH = "zhang_best_acc.pt"

for epoch in range(1, EPOCHS + 1):

    # Unfreeze backbone after FREEZE_EPOCHS
    if epoch == FREEZE_EPOCHS + 1:
        set_backbone_trainable(model, True)
        # Recreate optimizer so it now includes backbone params too
        opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WD)
        print(f" Unfroze backbone at epoch {epoch}. Trainable params:", count_trainable_params(model))

    tr_loss, tr_acc, tr_p, tr_r, tr_f1 = run_epoch(model, train_loader, train=True)
    te_loss, te_acc, te_p, te_r, te_f1 = run_epoch(model, test_loader,  train=False)

    row = dict(
        epoch=epoch,
        train_loss=tr_loss, train_acc=tr_acc, train_p=tr_p, train_r=tr_r, train_f1=tr_f1,
        test_loss=te_loss,  test_acc=te_acc,  test_p=te_p,  test_r=te_r,  test_f1=te_f1,
        bad_train_seen=train_ds.bad,
        bad_test_seen=test_ds.bad,
    )
    history.append(row)
    print(row)

    # Save best by F1
    if te_f1 > best_f1:
        best_f1 = te_f1
        torch.save({"model": model.state_dict(), "epoch": epoch, "metrics": row}, CKPT_F1_PATH)
        print("   saved best F1:", best_f1)

    # Save best by Accuracy
    if te_acc > best_acc:
        best_acc = te_acc
        torch.save({"model": model.state_dict(), "epoch": epoch, "metrics": row}, CKPT_ACC_PATH)
        print("   saved best ACC:", best_acc)

hist_df = pd.DataFrame(history)
hist_df.tail()


{'epoch': 1, 'train_loss': 1.9691552285637175, 'train_acc': 0.16726190476190475, 'train_p': 0.16936032481347948, 'train_r': 0.16726190476190478, 'train_f1': 0.16571191060619897, 'test_loss': 1.9251879709107536, 'test_acc': 0.18095238095238095, 'test_p': 0.22267547602086482, 'test_r': 0.18095238095238095, 'test_f1': 0.13963559483654125, 'bad_train_seen': 0, 'bad_test_seen': 0}
   saved best F1: 0.13963559483654125
   saved best ACC: 0.18095238095238095


{'epoch': 2, 'train_loss': 1.8654420226812363, 'train_acc': 0.24761904761904763, 'train_p': 0.24656349327214513, 'train_r': 0.2476190476190476, 'train_f1': 0.24119286287317684, 'test_loss': 1.914355184350695, 'test_acc': 0.19523809523809524, 'test_p': 0.22189500876297458, 'test_r': 0.19523809523809524, 'test_f1': 0.17836554431530582, 'bad_train_seen': 0, 'bad_test_seen': 0}
   saved best F1: 0.17836554431530582
   saved best ACC: 0.19523809523809524


{'epoch': 3, 'train_loss': 1.8246542470795768, 'train_acc': 0.2625, 'train_p': 0.2622635962541605, 'train_r': 0.26249999999999996, 'train_f1': 0.2590608876965276, 'test_loss': 1.9325144035475594, 'test_acc': 0.1976190476190476, 'test_p': 0.21733695353072477, 'test_r': 0.1976190476190476, 'test_f1': 0.16480470494281305, 'bad_train_seen': 0, 'bad_test_seen': 0}
   saved best ACC: 0.1976190476190476


{'epoch': 4, 'train_loss': 1.7819599700825555, 'train_acc': 0.30178571428571427, 'train_p': 0.30444368258965687, 'train_r': 0.30178571428571427, 'train_f1': 0.2983493330183263, 'test_loss': 1.9490567530904497, 'test_acc': 0.21666666666666667, 'test_p': 0.18642584559085876, 'test_r': 0.21666666666666665, 'test_f1': 0.1768158315740402, 'bad_train_seen': 0, 'bad_test_seen': 0}
   saved best ACC: 0.21666666666666667


{'epoch': 5, 'train_loss': 1.7519330978393555, 'train_acc': 0.32083333333333336, 'train_p': 0.3204611929825993, 'train_r': 0.32083333333333336, 'train_f1': 0.31908374880306145, 'test_loss': 2.0077152848243713, 'test_acc': 0.2, 'test_p': 0.2669402696019501, 'test_r': 0.19999999999999998, 'test_f1': 0.16532539036051094, 'bad_train_seen': 0, 'bad_test_seen': 0}
 Unfroze backbone at epoch 6. Trainable params: 16487495


...

{'epoch': 31, 'train_loss': 0.07287135023424136, 'train_acc': 0.9779761904761904, 'train_p': 0.9779895454223727, 'train_r': 0.9779761904761904, 'train_f1': 0.9779671352637952, 'test_loss': 3.7641971792493547, 'test_acc': 0.2119047619047619, 'test_p': 0.20957446590650722, 'test_r': 0.21190476190476187, 'test_f1': 0.20602148989338825, 'bad_train_seen': 0, 'bad_test_seen': 0}


{'epoch': 32, 'train_loss': 0.0771314081669386, 'train_acc': 0.9738095238095238, 'train_p': 0.9739144540185346, 'train_r': 0.9738095238095237, 'train_f1': 0.9738358553830244, 'test_loss': 3.629590937069484, 'test_acc': 0.22857142857142856, 'test_p': 0.2259250588493396, 'test_r': 0.22857142857142856, 'test_f1': 0.22348330859385937, 'bad_train_seen': 0, 'bad_test_seen': 0}


{'epoch': 33, 'train_loss': 0.07044750456615086, 'train_acc': 0.9779761904761904, 'train_p': 0.9779656697443526, 'train_r': 0.9779761904761904, 'train_f1': 0.9779648797390978, 'test_loss': 3.7383656672069003, 'test_acc': 0.2523809523809524, 'test_p': 0.25292247333716095, 'test_r': 0.25238095238095243, 'test_f1': 0.24829026274282742, 'bad_train_seen': 0, 'bad_test_seen': 0}


{'epoch': 34, 'train_loss': 0.08363350246716957, 'train_acc': 0.9732142857142857, 'train_p': 0.9732295728719464, 'train_r': 0.9732142857142858, 'train_f1': 0.9731988767290317, 'test_loss': 3.646328943116324, 'test_acc': 0.2357142857142857, 'test_p': 0.2562009168914722, 'test_r': 0.2357142857142857, 'test_f1': 0.22802519444132432, 'bad_train_seen': 0, 'bad_test_seen': 0}


KeyboardInterrupt: 

### 8) Final evaluation (best checkpoint)

In [21]:
def eval_checkpoint(path):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model"])
    model.eval()

    print("\n==============================")
    print("Loaded:", path)
    print("Epoch:", ckpt["epoch"])
    print("Snapshot metrics:", ckpt["metrics"])

    all_true, all_pred, all_files = [], [], []

    with torch.no_grad():
        for x, y, fn in tqdm(test_loader, leave=False):
            x = x.to(device)
            logits = model(x)
            pred = logits.argmax(1).cpu().numpy().tolist()

            all_pred.extend(pred)
            all_true.extend(y.numpy().tolist())
            all_files.extend(list(fn))

    print("\nFinal Evaluation:")
    print("Accuracy:", accuracy_score(all_true, all_pred))
    print(classification_report(all_true, all_pred,
                                target_names=Z_CLASSES,
                                digits=4,
                                zero_division=0))

    cm = confusion_matrix(all_true, all_pred)
    print("Confusion Matrix:\n", cm)

    return cm


# Evaluate best-by-accuracy model
cm_acc = eval_checkpoint("zhang_best_acc.pt")

# Evaluate best-by-F1 model
cm_f1  = eval_checkpoint("zhang_best_f1.pt")



Loaded: zhang_best_acc.pt
Epoch: 15
Snapshot metrics: {'epoch': 15, 'train_loss': 0.11880094558000565, 'train_acc': 0.9619047619047619, 'train_p': 0.961973562185956, 'train_r': 0.9619047619047619, 'train_f1': 0.9619047075363664, 'test_loss': 3.2291485582079207, 'test_acc': 0.27380952380952384, 'test_p': 0.27527069839099916, 'test_r': 0.27380952380952384, 'test_f1': 0.2695820980722038, 'bad_train_seen': 0, 'bad_test_seen': 0}



Final Evaluation:
Accuracy: 0.27380952380952384
              precision    recall  f1-score   support

   happiness     0.2444    0.1833    0.2095        60
satisfaction     0.3095    0.2167    0.2549        60
    surprise     0.2576    0.2833    0.2698        60
     sadness     0.3000    0.3000    0.3000        60
       anger     0.2917    0.2333    0.2593        60
     disgust     0.2500    0.2667    0.2581        60
        fear     0.2737    0.4333    0.3355        60

    accuracy                         0.2738       420
   macro avg     0.2753    0.2738    0.2696       420
weighted avg     0.2753    0.2738    0.2696       420

Confusion Matrix:
 [[11  4 12  8  5 11  9]
 [ 9 13  5  5  8 10 10]
 [ 3  5 17  5  6  7 17]
 [ 6  5  4 18  7  9 11]
 [ 4  4 11  7 14  6 14]
 [ 7  5  9 13  2 16  8]
 [ 5  6  8  4  6  5 26]]

Loaded: zhang_best_f1.pt
Epoch: 15
Snapshot metrics: {'epoch': 15, 'train_loss': 0.11880094558000565, 'train_acc': 0.9619047619047619, 'train_p': 0.961973562185956, 


Final Evaluation:
Accuracy: 0.27380952380952384
              precision    recall  f1-score   support

   happiness     0.2444    0.1833    0.2095        60
satisfaction     0.3095    0.2167    0.2549        60
    surprise     0.2576    0.2833    0.2698        60
     sadness     0.3000    0.3000    0.3000        60
       anger     0.2917    0.2333    0.2593        60
     disgust     0.2500    0.2667    0.2581        60
        fear     0.2737    0.4333    0.3355        60

    accuracy                         0.2738       420
   macro avg     0.2753    0.2738    0.2696       420
weighted avg     0.2753    0.2738    0.2696       420

Confusion Matrix:
 [[11  4 12  8  5 11  9]
 [ 9 13  5  5  8 10 10]
 [ 3  5 17  5  6  7 17]
 [ 6  5  4 18  7  9 11]
 [ 4  4 11  7 14  6 14]
 [ 7  5  9 13  2 16  8]
 [ 5  6  8  4  6  5 26]]


In [22]:
import numpy as np
from PIL import Image, ImageSequence

def load_gif_frames_full(path, max_frames=200):
    frames = []
    im = Image.open(path)
    for i, fr in enumerate(ImageSequence.Iterator(im)):
        if max_frames is not None and i >= max_frames:
            break
        frames.append(fr.convert("RGB"))
    return frames

def make_clip_starts(n_frames, seq_len=4, n_clips=5):
    """
    Return clip start indices for evenly-spaced clips.
    Ensures each clip has seq_len frames (pad handled later).
    """
    if n_frames <= 1:
        return [0] * n_clips

    # If GIF is too short, all clips start at 0
    if n_frames <= seq_len:
        return [0] * n_clips

    max_start = n_frames - seq_len
    # Evenly spaced starts from 0..max_start
    starts = np.linspace(0, max_start, n_clips).round().astype(int).tolist()
    return starts

def get_clip(frames, start, seq_len=4):
    """
    Extract seq_len consecutive frames from 'start'; if out of range, repeat last frame.
    """
    if len(frames) == 0:
        return []
    clip = []
    for i in range(seq_len):
        idx = start + i
        if idx < len(frames):
            clip.append(frames[idx])
        else:
            clip.append(frames[-1])
    return clip


In [23]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tqdm import tqdm
import os
import torch

@torch.no_grad()
def eval_singleclip_from_df(df_eval, transform, ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model"])
    model.eval()

    y_true, y_pred = [], []
    for _, r in tqdm(df_eval.iterrows(), total=len(df_eval), leave=False):
        path = r["gif_path"]
        lab  = r["zhang_label"]

        frames = load_gif_frames(path, max_frames=60)
        frames = sample_or_pad(frames, seq_len=SEQ_LEN)
        x = torch.stack([transform(fr) for fr in frames], dim=0).unsqueeze(0).to(device)  # [1,T,3,224,224]

        logits = model(x)
        pred = int(logits.argmax(1).item())

        y_true.append(label2id[lab])
        y_pred.append(pred)

    acc = accuracy_score(y_true, y_pred)
    rep = classification_report(y_true, y_pred, target_names=Z_CLASSES, digits=4, zero_division=0)
    cm  = confusion_matrix(y_true, y_pred)
    return acc, rep, cm

@torch.no_grad()
def eval_multiclip_from_df(df_eval, transform, ckpt_path, n_clips=5, max_frames=200):
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model"])
    model.eval()

    y_true, y_pred = [], []
    for _, r in tqdm(df_eval.iterrows(), total=len(df_eval), leave=False):
        path = r["gif_path"]
        lab  = r["zhang_label"]

        frames = load_gif_frames_full(path, max_frames=max_frames)
        if len(frames) == 0:
            # fall back to zeros
            y_true.append(label2id[lab])
            y_pred.append(0)
            continue

        starts = make_clip_starts(len(frames), seq_len=SEQ_LEN, n_clips=n_clips)

        # accumulate logits
        logits_sum = None
        for s in starts:
            clip = get_clip(frames, s, seq_len=SEQ_LEN)
            x = torch.stack([transform(fr) for fr in clip], dim=0).unsqueeze(0).to(device)  # [1,T,3,224,224]
            logits = model(x)  # [1,7]
            logits_sum = logits if logits_sum is None else (logits_sum + logits)

        logits_avg = logits_sum / len(starts)
        pred = int(logits_avg.argmax(1).item())

        y_true.append(label2id[lab])
        y_pred.append(pred)

    acc = accuracy_score(y_true, y_pred)
    rep = classification_report(y_true, y_pred, target_names=Z_CLASSES, digits=4, zero_division=0)
    cm  = confusion_matrix(y_true, y_pred)
    return acc, rep, cm


# -------------------------
# Run comparison on the SAME checkpoint
# -------------------------
CKPT = "zhang_best_acc.pt"   # or "zhang_best_f1.pt"

print("\n=== Single-clip evaluation ===")
acc1, rep1, cm1 = eval_singleclip_from_df(test_df, test_tf, CKPT)
print("Accuracy:", acc1)
print(rep1)
print("Confusion Matrix:\n", cm1)

print("\n=== Multi-clip evaluation (n_clips=5) ===")
acc2, rep2, cm2 = eval_multiclip_from_df(test_df, test_tf, CKPT, n_clips=5, max_frames=200)
print("Accuracy:", acc2)
print(rep2)
print("Confusion Matrix:\n", cm2)



=== Single-clip evaluation ===


Accuracy: 0.27380952380952384
              precision    recall  f1-score   support

   happiness     0.2444    0.1833    0.2095        60
satisfaction     0.3095    0.2167    0.2549        60
    surprise     0.2576    0.2833    0.2698        60
     sadness     0.3000    0.3000    0.3000        60
       anger     0.2917    0.2333    0.2593        60
     disgust     0.2500    0.2667    0.2581        60
        fear     0.2737    0.4333    0.3355        60

    accuracy                         0.2738       420
   macro avg     0.2753    0.2738    0.2696       420
weighted avg     0.2753    0.2738    0.2696       420

Confusion Matrix:
 [[11  4 12  8  5 11  9]
 [ 9 13  5  5  8 10 10]
 [ 3  5 17  5  6  7 17]
 [ 6  5  4 18  7  9 11]
 [ 4  4 11  7 14  6 14]
 [ 7  5  9 13  2 16  8]
 [ 5  6  8  4  6  5 26]]

=== Multi-clip evaluation (n_clips=5) ===


Accuracy: 0.26904761904761904
              precision    recall  f1-score   support

   happiness     0.2500    0.2167    0.2321        60
satisfaction     0.2826    0.2167    0.2453        60
    surprise     0.2222    0.2000    0.2105        60
     sadness     0.2931    0.2833    0.2881        60
       anger     0.2979    0.2333    0.2617        60
     disgust     0.2462    0.2667    0.2560        60
        fear     0.2857    0.4667    0.3544        60

    accuracy                         0.2690       420
   macro avg     0.2682    0.2690    0.2640       420
weighted avg     0.2682    0.2690    0.2640       420

Confusion Matrix:
 [[13  5  8  9  6 10  9]
 [ 9 13  2  6  9 11 10]
 [ 5  7 12  5  4  9 18]
 [ 6  5  6 17  5 10 11]
 [ 4  6 11  7 14  5 13]
 [ 8  5  9 10  3 16  9]
 [ 7  5  6  4  6  4 28]]


In [29]:
LR = 1e-4
WD = 1e-4
EPOCHS = 30

# IMPORTANT: create optimizer AFTER freezing so it only optimizes trainable params at first
opt = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR,
    weight_decay=WD
)


def run_epoch(model, loader, train=True):
    model.train(train)
    losses = []
    y_true, y_pred = [], []

    for x, y, _fn in tqdm(loader, leave=False):
        x, y = x.to(device), y.to(device)

        if train:
            opt.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train):
            logits = model(x)
            loss = F.cross_entropy(logits, y)

            if train:
                loss.backward()
                opt.step()

        losses.append(loss.item())
        y_true.extend(y.detach().cpu().numpy().tolist())
        y_pred.extend(logits.argmax(1).detach().cpu().numpy().tolist())

    acc = accuracy_score(y_true, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    return float(np.mean(losses)), float(acc), float(p), float(r), float(f1)

history = []

best_f1  = -1.0
best_acc = -1.0

CKPT_F1_PATH  = "zhang_best_f1.pt"
CKPT_ACC_PATH = "zhang_best_acc.pt"

for epoch in range(1, EPOCHS + 1):

    # Unfreeze backbone after FREEZE_EPOCHS
    if epoch == FREEZE_EPOCHS + 1:
        set_backbone_trainable(model, True)

        # Differential LR: backbone slower to prevent overfitting / forgetting
        opt = torch.optim.Adam([
            {"params": model.backbone.parameters(), "lr": 1e-5},
            {"params": model.convgru.parameters(),  "lr": 1e-4},
            {"params": model.fc.parameters(),       "lr": 1e-4},
        ], weight_decay=WD)

        print(f" Unfroze backbone at epoch {epoch}. Using differential LR (backbone=1e-5, head=1e-4).")


    tr_loss, tr_acc, tr_p, tr_r, tr_f1 = run_epoch(model, train_loader, train=True)
    te_loss, te_acc, te_p, te_r, te_f1 = run_epoch(model, test_loader,  train=False)

    row = dict(
        epoch=epoch,
        train_loss=tr_loss, train_acc=tr_acc, train_p=tr_p, train_r=tr_r, train_f1=tr_f1,
        test_loss=te_loss,  test_acc=te_acc,  test_p=te_p,  test_r=te_r,  test_f1=te_f1,
        bad_train_seen=train_ds.bad,
        bad_test_seen=test_ds.bad,
    )
    history.append(row)
    print(row)

    # Save best by F1
    if te_f1 > best_f1:
        best_f1 = te_f1
        torch.save({"model": model.state_dict(), "epoch": epoch, "metrics": row}, CKPT_F1_PATH)
        print("   saved best F1:", best_f1)

    # Save best by Accuracy
    if te_acc > best_acc:
        best_acc = te_acc
        torch.save({"model": model.state_dict(), "epoch": epoch, "metrics": row}, CKPT_ACC_PATH)
        print("   saved best ACC:", best_acc)

hist_df = pd.DataFrame(history)
hist_df.tail()


{'epoch': 1, 'train_loss': 1.9752891233989172, 'train_acc': 0.1732142857142857, 'train_p': 0.17536108550911347, 'train_r': 0.1732142857142857, 'train_f1': 0.17219123296892397, 'test_loss': 1.910635565008436, 'test_acc': 0.20238095238095238, 'test_p': 0.3032351194640219, 'test_r': 0.20238095238095238, 'test_f1': 0.16858616269702992, 'bad_train_seen': 0, 'bad_test_seen': 0}
   saved best F1: 0.16858616269702992
   saved best ACC: 0.20238095238095238


{'epoch': 2, 'train_loss': 1.9290172840867723, 'train_acc': 0.19642857142857142, 'train_p': 0.1883496758241548, 'train_r': 0.19642857142857142, 'train_f1': 0.18796855800313825, 'test_loss': 1.9113875372069222, 'test_acc': 0.20238095238095238, 'test_p': 0.22204211676916924, 'test_r': 0.20238095238095236, 'test_f1': 0.1655162273845617, 'bad_train_seen': 0, 'bad_test_seen': 0}


{'epoch': 3, 'train_loss': 1.8897717829261507, 'train_acc': 0.2220238095238095, 'train_p': 0.21943806226113732, 'train_r': 0.22202380952380948, 'train_f1': 0.21645396453347532, 'test_loss': 1.906955565725054, 'test_acc': 0.21428571428571427, 'test_p': 0.19937343743987188, 'test_r': 0.21428571428571427, 'test_f1': 0.16790455770394094, 'bad_train_seen': 0, 'bad_test_seen': 0}
   saved best ACC: 0.21428571428571427


{'epoch': 4, 'train_loss': 1.8812306821346283, 'train_acc': 0.22976190476190475, 'train_p': 0.22611335493443008, 'train_r': 0.22976190476190478, 'train_f1': 0.22338992380195455, 'test_loss': 1.922827192715236, 'test_acc': 0.1976190476190476, 'test_p': 0.16884684406002945, 'test_r': 0.1976190476190476, 'test_f1': 0.1370078228834797, 'bad_train_seen': 0, 'bad_test_seen': 0}


{'epoch': 5, 'train_loss': 1.838101714849472, 'train_acc': 0.2630952380952381, 'train_p': 0.25784289713618935, 'train_r': 0.2630952380952381, 'train_f1': 0.2564974280650704, 'test_loss': 1.9118260145187378, 'test_acc': 0.21666666666666667, 'test_p': 0.2330643030269896, 'test_r': 0.21666666666666665, 'test_f1': 0.17753969493660215, 'bad_train_seen': 0, 'bad_test_seen': 0}
   saved best F1: 0.17753969493660215
   saved best ACC: 0.21666666666666667
 Unfroze backbone at epoch 6. Using differential LR (backbone=1e-5, head=1e-4).


...

{'epoch': 20, 'train_loss': 0.3907875046133995, 'train_acc': 0.881547619047619, 'train_p': 0.8818699746163737, 'train_r': 0.881547619047619, 'train_f1': 0.8816136300117546, 'test_loss': 2.8985525369644165, 'test_acc': 0.21904761904761905, 'test_p': 0.21839381879268177, 'test_r': 0.21904761904761907, 'test_f1': 0.21040480231747288, 'bad_train_seen': 0, 'bad_test_seen': 0}


{'epoch': 21, 'train_loss': 0.3789957288120474, 'train_acc': 0.8732142857142857, 'train_p': 0.8735227798767621, 'train_r': 0.8732142857142858, 'train_f1': 0.8732765852653779, 'test_loss': 2.9160889046532765, 'test_acc': 0.23809523809523808, 'test_p': 0.25961073800042034, 'test_r': 0.2380952380952381, 'test_f1': 0.23558812585010644, 'bad_train_seen': 0, 'bad_test_seen': 0}


{'epoch': 22, 'train_loss': 0.31517884268292357, 'train_acc': 0.9005952380952381, 'train_p': 0.9006824879808804, 'train_r': 0.900595238095238, 'train_f1': 0.9005514229144383, 'test_loss': 3.0275993858064925, 'test_acc': 0.2119047619047619, 'test_p': 0.21020396934476485, 'test_r': 0.2119047619047619, 'test_f1': 0.20627792058488387, 'bad_train_seen': 0, 'bad_test_seen': 0}


{'epoch': 23, 'train_loss': 0.3010478485375643, 'train_acc': 0.9077380952380952, 'train_p': 0.9079194746455378, 'train_r': 0.9077380952380951, 'train_f1': 0.9077507208722894, 'test_loss': 3.0899765150887624, 'test_acc': 0.22380952380952382, 'test_p': 0.2321203462185502, 'test_r': 0.22380952380952382, 'test_f1': 0.21776632589010272, 'bad_train_seen': 0, 'bad_test_seen': 0}


KeyboardInterrupt: 

### 9) Save outputs (history + test predictions)

In [ ]:
hist_df.to_csv("training_history.csv", index=False)

pred_df = pd.DataFrame({
    "file": all_files,
    "y_true": [id2label[i] for i in all_true],
    "y_pred": [id2label[i] for i in all_pred],
})
pred_df.to_csv("test_predictions.csv", index=False)

print("Saved: training_history.csv, test_predictions.csv, and", CKPT_PATH)
pred_df.head()
